# FIRELINE — Prioritas area & dashboard yang dapat dijelaskan

Notebook 06 mengubah hasil audit menjadi dashboard fitur **Mapping / Peta Prioritas Titik Api**. Mulai dari pertanyaan pengguna: **area mana yang perlu lebih diperhatikan pada tanggal yang saya pilih, dan mengapa?**

Pipeline lengkap berjalan di sel pertama sesudah konfigurasi. Sel-sel berikut memeriksa hasil dan menjelaskan keputusan. Skor ini adalah heuristik untuk prototipe historis; label kejadian terverifikasi belum tersedia.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
ROOT = Path.cwd() if (Path.cwd() / "fireline").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from fireline.pipeline import settings, audit, build
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})
config = settings(ROOT / "config.local.json")

In [ ]:
summary, grids = build(ROOT / "config.local.json")
display(Markdown(f"**{summary['kalimantan_observations']:,} observasi → {summary['grid_day_rows']:,} grid-hari; {summary['unique_grids']:,} grid unik.**"))
display(pd.DataFrame(summary["stage_counts"]))

## 1. Mengelompokkan observasi menjadi area

Grid 0,1° adalah kotak bantu sekitar 11 km per sisi di sekitar khatulistiwa. Satu kunci `grid_id + date` berarti satu kotak pada satu tanggal UTC. Ini bukan pemisahan kejadian kebakaran yang telah divalidasi.

Marker ditempatkan pada **observasi dengan FRP tertinggi dalam grid-hari**, bukan pusat grid. Jarak sekolah dihitung dari marker representatif itu. Jadi angka jaraknya tidak bisa dibandingkan langsung dengan median seluruh observasi atau median pusat grid pada baseline lama.

Grid yang melewati batas provinsi diberi catatan. Untuk filter, label provinsi mengikuti observasi representatif; hitungan provinsi di audit tetap dihitung dari setiap observasi asli.

In [ ]:
cols=["grid_id","date","province_name","region_name","detection_count","latitude","longitude","active_days_7d","detections_7d","frp_median_mw"]
display(grids[cols].head(12))
assert not grids.duplicated(["grid_id","date"]).any()
assert grids.detection_count.sum() == summary["kalimantan_observations"]
assert grids.active_days_7d.between(1,7).all()

## 2. Tiga pertanyaan yang membentuk skor

1. **Aktivitas:** seberapa banyak observasi di area itu pada hari terpilih? Diubah menjadi persentil di antara semua area aktif Kalimantan pada hari yang sama.
2. **Persistensi:** berapa tanggal berbeda area ini muncul dalam tujuh hari termasuk hari terpilih? Dibagi 7; pengamatan berulang tidak otomatis membuktikan api menyala terus.
3. **FRP:** seberapa tinggi nilai tengah radiasi panas area itu dibanding area lain pada hari yang sama?

`skor = 40 × aktivitas + 35 × persistensi + 25 × FRP`.

Perubahan dari baseline v2: persistensi memakai **hari aktif/7**, bukan persentil total deteksi tujuh hari. Ini membedakan frekuensi hari dari volume observasi, walaupun keduanya masih dapat berkorelasi. Mid-rank mencegah satu area aktif otomatis mendapat persentil 100. Skor nol/tidak terisi bukan pernyataan wilayah aman.

Batas awal: P1 ≥80, P2 60–<80, P3 <60. Batas ini asumsi desain, bukan batas bahaya yang tervalidasi. Skor relatif harian tidak dipakai untuk membuktikan tren risiko antartanggal. Filter provinsi/P1 menyaring tampilan tanpa menghitung ulang pembanding.

In [ ]:
demo = grids.loc[grids.date.eq(summary["demo_date"])].sort_values(["priority_score","grid_id"], ascending=[False,True])
display(demo[["grid_id","region_name","priority_level","priority_score","activity_points","persistence_points","intensity_points"]].head(10))
assert np.allclose(grids.priority_score, grids.activity_points + grids.persistence_points + grids.intensity_points, atol=0.0001)
fig, ax = plt.subplots(figsize=(11,4))
top = demo.head(10).set_index("grid_id")[["activity_points","persistence_points","intensity_points"]]
top.plot.barh(stacked=True, ax=ax, color=["#244d40","#94aa6f","#d6bd62"])
ax.invert_yaxis(); ax.set_xlim(0,100); ax.set_xlabel("Kontribusi terhadap skor, poin")
ax.set_title("Mengapa sepuluh area ini mendapat prioritas? " + summary["demo_date"])
ax.legend(["Aktivitas","Hari aktif","FRP"], loc="lower right")
plt.tight_layout(); plt.show()

## 3. Apakah fitur saling berhubungan?

Korelasi Spearman mengukur apakah nilai yang lebih tinggi pada satu fitur cenderung beriringan dengan fitur lain. Mendekati +1 berarti urutannya cenderung sama, mendekati -1 berarti berlawanan, dan mendekati 0 berarti tidak ada hubungan monoton yang kuat dalam sampel ini.

Tabel ini bersifat deskriptif atas grid-hari, dengan pengamatan berulang dari area yang sama. Korelasi tidak membuktikan penyebab atau validitas skor. Kami tidak menghitung korelasi cuaca yang tanggalnya digeser.

In [ ]:
corr_cols=["detection_count","active_days_7d","frp_median_mw"]
correlation=grids[corr_cols].corr(method="spearman")
display(correlation.round(3))
fig, ax=plt.subplots(figsize=(6,4))
im=ax.imshow(correlation,vmin=-1,vmax=1,cmap="BrBG")
ax.set_xticks(range(3),["Observasi hari ini","Hari aktif / 7","FRP median"],rotation=15)
ax.set_yticks(range(3),["Observasi hari ini","Hari aktif / 7","FRP median"])
for i in range(3):
    for j in range(3): ax.text(j,i,f"{correlation.iloc[i,j]:.2f}",ha="center",va="center")
fig.colorbar(im,ax=ax,label="Spearman");plt.tight_layout();plt.show()

## 4. Uji bobot: apakah Top 10 mudah berubah?

Kami membandingkan 40/35/25 dengan bobot sama, penekanan aktivitas, penekanan persistensi, dan penghapusan satu fitur. Jaccard adalah anggota yang sama dibagi seluruh anggota unik dari dua daftar. 1 berarti daftar sama; 0 berarti tidak ada anggota bersama.

Hanya hari dengan lebih dari 10 area yang masuk ringkasan: jika areanya cuma 5, dua daftar Top 10 pasti memuat kelimanya dan terlihat stabil secara semu. Uji ini menilai kestabilan, belum mengukur akurasi atau manfaat lapangan.

In [ ]:
sensitivity_table = pd.DataFrame(summary["sensitivity"])
display(sensitivity_table.drop(columns="weights"))
fig, ax=plt.subplots(figsize=(10,3.5))
ax.barh(sensitivity_table.variant, sensitivity_table.mean_top10_jaccard, color="#517950")
ax.set_xlim(0,1);ax.set_xlabel("Rata-rata Jaccard Top 10 (0–1)")
ax.set_title("Kestabilan anggota ketika bobot berubah")
plt.tight_layout();plt.show()
display(grids.priority_level.value_counts().to_frame("grid_days"))

## 5. Sekolah dan batas kab/kota sebagai konteks

School distance tidak masuk rumus. Kedekatan geografis belum membuktikan paparan asap, anak-anak terdampak, atau sekolah beroperasi. Boundary kab/kota dipulihkan dari HTML Kaka; seluruh skor, confidence agregat, dan ranking lama dibuang. Sumber/tahun boundary kab/kota perlu dikonfirmasi sebelum publikasi final.

In [ ]:
display(grids[["grid_id","date","nearest_school_name","nearest_school_km","quality_note"]].head(8))
print("Sekolah yang masuk poligon:", summary["schools_in_boundary"])
print("Median jarak observasi representatif per grid-hari:", summary["median_representative_school_km"],"km")
print("Grid-hari tanpa label kab/kota:", summary["unmapped_representative_grid_days"])
print(summary["region_note"])

## 6. Dashboard dan keterlacakan hasil

Dashboard HTML dan Streamlit memakai payload yang sama. Top 10 pada daftar, nomor marker, dan detail berasal dari pemilahan data yang sama. Tanggal terpilih adalah filter satu hari; konteks persistensi/tren memakai tujuh hari terakhir.

Snapshot historis yang dibuka sebagai demo dipilih pada hari observasi terbanyak. Waktu pembangunan file dan waktu observasi ditampilkan terpisah. Pengiriman notifikasi, Community, dan verifikasi laporan tetap mengikuti keputusan produk terpisah dan tidak dijalankan di sini.

In [ ]:
artifact = ROOT / "review_outputs" / "FIRELINE_Priority_Map.html"
assert artifact.exists()
display(Markdown("[Buka dashboard HTML](../review_outputs/FIRELINE_Priority_Map.html)"))
display(pd.DataFrame(summary["sources"]).T[["file","sha256"]])
print("Ukuran dashboard:",round(artifact.stat().st_size/1024/1024,2),"MB")
print("Selesai. Dashboard dapat dibuka langsung atau melalui Streamlit.")

## Pertanyaan yang tetap dibawa saat review

- PMA/UX mengonfirmasi apakah istilah area-grid diterima sebagai representasi titik prioritas; grid 0,1° belum kalibrasi resolusi operasional.
- Bobot dan ambang kategori perlu masukan domain serta uji dengan label lapangan.
- Estimasi dampak belum tersedia. Cuaca/populasi tidak boleh diisi dengan data yang tidak sesuai ruang/waktu.
- Data utama tetap sumber resmi observasi; sumber tambahan menjadi konteks yang keterbatasannya dicatat.

Sumber: [NASA FIRMS](https://firms.modaps.eosdis.nasa.gov/active_fire/), [BIG](https://geoservices.big.go.id/gis/rest/services/STIG/Batas_Provinsi/MapServer), [referensi sekolah](https://www.kaggle.com/datasets/marchotridyo/indonesia-school-dataset-with-province-data).